This notebook compares the VLE of isopropanol-water:
* using group-contribution (GCM/Constantinou-Gani) calculated pure properties
* using corrected (experimental) pure properties for isopropanol

This mirrors `case_studies/acetone_methanol.ipynb`, and highlights how sensitive the predicted
VLE behaviour (and any azeotrope) is to the accuracy of the pure-component property estimates,
same as observed for THF-water and ethanol-water in `tutorials/`.

In [ ]:
# imports to enable the notebook to access the src folder
import sys
import os
sys.path.append(os.path.dirname(os.path.dirname(os.getcwd())))

# import the necessary packages
from src.molecules import StructuralValidator
from src.pure import PropertyCalculator
from src.pure import AntoineCalculator
from src.data import load_unifac_data
from src.mixture.unifac.pyomo_vle import pyo_vle
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# step1: define the compounds
mixture = {'isopropanol': {'CH3': 2, 'CH': 1, 'OH': 1},
           'water': {'H2O': 1}
           }
# validate the structures
validator = StructuralValidator()
validator.validate(mixture)

In [ ]:
# step2: calculate the required properties (GCM / Constantinou-Gani)
prop_list = ['tc', 'pc', 'tb', 'ac', 'hv']
pure_calc = PropertyCalculator()
pure_props = pure_calc.calculate(mixture, prop_list)

print(pure_props)

# step3: calculate the Antoine coefficients
antoine_params = {}
antoine_calc = AntoineCalculator()
for compound, props in pure_props.items():
    results = antoine_calc.calculate_antoine_parameters(mixture, critical_properties=props, method='enthalpy_vap')
    print('optimization results was', results['termination_condition'])
    antoine_params[compound] = {}
    antoine_params[compound]['A'] = results['A']
    antoine_params[compound]['B'] = results['B']
    antoine_params[compound]['C'] = results['C']


# step4: perform the vle
# define number of data points
n_data = 100
# set composition
x1 = np.linspace(0, 1, n_data)  # Component 1 (isopropanol): 0 to 1
x2 = 1 - x1                     # Component 2 (water): 1 to 0 (complement)
x = np.column_stack([x1, x2])   # Combine into 2D array
# provide initial guess for temperature
# Extract boiling points
boiling_points = [props['tb'] for props in pure_props.values()]
T_min = min(boiling_points)
T_max = max(boiling_points)
T0 = np.linspace(T_min, T_max, n_data)
# Load all UNIFAC parameters
rk_data, qk_data, interaction_data = load_unifac_data()
# perform the VLE calculation
results, model = pyo_vle(x, T0, mixture, rk_data, qk_data, interaction_data, antoine_params, verbose=False)

gamma_gcm = np.empty_like(x)
T_gcm = np.empty_like(T0)
y_gcm = np.empty_like(x)
psat_gcm = np.empty_like(x)

for i in model.gamma.keys():
    gamma_gcm[i] = model.gamma[i].value

for i in model.T.keys():
    T_gcm[i] = model.T[i].value

for i in model.y.keys():
    y_gcm[i] = model.y[i].value

for i in model.psat.keys():
    psat_gcm[i] = model.psat[i].value

compute the azeotrope predicted with GCM (uncorrected) properties

In [ ]:
# compute the azeotrope using GCM (uncorrected) properties/Antoine
from src.mixture.unifac.pyomo_azeo import pyo_azeo
T0_az = np.array((T_min + T_max) / 2)
x0_az = np.array([[0.5, 0.5]])
results_az_gcm, model_az_gcm = pyo_azeo(x0_az, T0_az, mixture, rk_data, qk_data, interaction_data,
                                         antoine_params, verbose=False)

In [ ]:
# retrieving the solution
gamma_az_gcm = np.empty_like(x0_az)
T_az_gcm = np.empty_like(T0_az)
y_az_gcm = np.empty_like(x0_az)
x_az_gcm = np.empty_like(x0_az)
psat_az_gcm = np.empty_like(x0_az)

for i in model_az_gcm.gamma.keys():
    gamma_az_gcm[i] = model_az_gcm.gamma[i].value

for i in model_az_gcm.T.keys():
    T_az_gcm = model_az_gcm.T[i].value

for i in model_az_gcm.y.keys():
    y_az_gcm[i] = model_az_gcm.y[i].value

for i in model_az_gcm.x.keys():
    x_az_gcm[i] = model_az_gcm.x[i].value

for i in model_az_gcm.psat.keys():
    psat_az_gcm[i] = model_az_gcm.psat[i].value

print('GCM azeotrope found at T=', T_az_gcm, 'K')
print('Liquid composition:', x_az_gcm)
print('Vapor composition:', y_az_gcm)

now we fix the properties of isopropanol using experimental data
(same values as used in `config/IECR/screen_isopropanol_water.yaml`)

In [ ]:
# fix the properties of isopropanol
pure_props['isopropanol']['tc'] = 509.2
pure_props['isopropanol']['pc'] = 49.0
pure_props['isopropanol']['tb'] = 355.5

# step3: recalculate the Antoine coefficients with corrected properties
antoine_params = {}
antoine_calc = AntoineCalculator()
for compound, props in pure_props.items():
    results = antoine_calc.calculate_antoine_parameters(mixture, critical_properties=props, method='enthalpy_vap')
    print('optimization results was', results['termination_condition'])
    antoine_params[compound] = {}
    antoine_params[compound]['A'] = results['A']
    antoine_params[compound]['B'] = results['B']
    antoine_params[compound]['C'] = results['C']

# perform the VLE calculation with corrected properties
results, model = pyo_vle(x, T0, mixture, rk_data, qk_data, interaction_data, antoine_params, verbose=False)

gamma_exp = np.empty_like(x)
T_exp = np.empty_like(T0)
y_exp = np.empty_like(x)
psat_exp = np.empty_like(x)

for i in model.gamma.keys():
    gamma_exp[i] = model.gamma[i].value

for i in model.T.keys():
    T_exp[i] = model.T[i].value

for i in model.y.keys():
    y_exp[i] = model.y[i].value

for i in model.psat.keys():
    psat_exp[i] = model.psat[i].value

compute the azeotrope predicted with corrected (experimental) properties

In [ ]:
# compute the azeotrope using corrected (experimental) properties/Antoine
T0_az = np.array((T_min + T_max) / 2)
x0_az = np.array([[0.5, 0.5]])
results_az_exp, model_az_exp = pyo_azeo(x0_az, T0_az, mixture, rk_data, qk_data, interaction_data,
                                         antoine_params, verbose=False)

In [ ]:
# retrieving the solution
gamma_az_exp = np.empty_like(x0_az)
T_az_exp = np.empty_like(T0_az)
y_az_exp = np.empty_like(x0_az)
x_az_exp = np.empty_like(x0_az)
psat_az_exp = np.empty_like(x0_az)

for i in model_az_exp.gamma.keys():
    gamma_az_exp[i] = model_az_exp.gamma[i].value

for i in model_az_exp.T.keys():
    T_az_exp = model_az_exp.T[i].value

for i in model_az_exp.y.keys():
    y_az_exp[i] = model_az_exp.y[i].value

for i in model_az_exp.x.keys():
    x_az_exp[i] = model_az_exp.x[i].value

for i in model_az_exp.psat.keys():
    psat_az_exp[i] = model_az_exp.psat[i].value

print('Corrected azeotrope found at T=', T_az_exp, 'K')
print('Liquid composition:', x_az_exp)
print('Vapor composition:', y_az_exp)

load experimental VLE data for isopropanol-water
(`data/vle_exp.xlsx`, sheet `Data_2`)

In [ ]:
# load experimental VLE data
import pandas as pd
df = pd.read_excel('../../data/vle_exp.xlsx', sheet_name='Data_2')
df

we make a combined T-x-y and x-y diagram, (a) and (b)

In [ ]:
# Combined T-x-y (a) and x-y (b) diagram: GCM vs corrected (experimental) properties
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6), dpi=150)

# ========== (a) T-x-y diagram ==========
ax1.plot(x[:, 0], T_gcm, color='#2ca02c', linewidth=2.5, linestyle='--',
         label='UNIFAC (GC properties)')
ax1.plot(y_gcm[:, 0], T_gcm, color='#2ca02c', linewidth=2.5, linestyle='--')

ax1.plot(x[:, 0], T_exp, color='#1f77b4', linewidth=2.5,
         label='UNIFAC (exp. properties)')
ax1.plot(y_exp[:, 0], T_exp, color='#1f77b4', linewidth=2.5)

if df is not None:
    ax1.scatter(df['x'], df['T'], s=80, facecolors='none', edgecolors='#d62728',
                linewidths=2, marker='o', label='Experimental', zorder=5)
    ax1.scatter(df['y'], df['T'], s=80, facecolors='none', edgecolors='#d62728',
                linewidths=2, marker='o', zorder=5)

ax1.scatter(x_az_gcm[0, 0], T_az_gcm, s=160, color='#2ca02c', marker='*',
            edgecolors='black', linewidths=1, zorder=6,
            label=f'Azeotrope (GC): x={x_az_gcm[0, 0]:.3f}, T={T_az_gcm:.1f} K')
ax1.scatter(x_az_exp[0, 0], T_az_exp, s=160, color='#1f77b4', marker='*',
            edgecolors='black', linewidths=1, zorder=6,
            label=f'Azeotrope (exp): x={x_az_exp[0, 0]:.3f}, T={T_az_exp:.1f} K')

x_az_lit, T_az_lit = 0.6740, 353.70
ax1.scatter(x_az_lit, T_az_lit, s=160, color='#ffd700', marker='*',
            edgecolors='black', linewidths=1, zorder=6,
            label=f'Azeotrope (literature): x={x_az_lit:.3f}, T={T_az_lit:.1f} K')

ax1.set_xlabel('Isopropanol mole fraction, $x_{\\mathrm{IPA}}$, $y_{\\mathrm{IPA}}$ [-]', fontsize=13)
ax1.set_ylabel('Temperature [K]', fontsize=13)
ax1.set_xlim(-0.02, 1.02)
ax1.tick_params(axis='both', which='major', labelsize=11, direction='in',
                length=6, width=1.5, top=True, right=True)
ax1.legend(loc='upper right', fontsize=8, frameon=True, fancybox=False,
           edgecolor='black', framealpha=1)
ax1.grid(True, linestyle='--', linewidth=0.5, alpha=0.3, color='gray')
ax1.text(0.02, 0.05, '(a)', transform=ax1.transAxes, fontsize=14,
         fontweight='bold', va='bottom', ha='left',
         bbox=dict(boxstyle='round,pad=0.3', facecolor='white', edgecolor='black', linewidth=1))

# ========== (b) x-y diagram ==========
ax2.plot([0, 1], [0, 1], 'k--', linewidth=1.5, alpha=0.6, label='$y = x$', zorder=1)

ax2.plot(x[:, 0], y_gcm[:, 0], color='#2ca02c', linewidth=2.5,
         linestyle='--', label='UNIFAC (GC properties)', zorder=3)
ax2.plot(x[:, 0], y_exp[:, 0], color='#1f77b4', linewidth=2.5,
         label='UNIFAC (exp. properties)', zorder=4)

if df is not None:
    ax2.scatter(df['x'], df['y'], s=80, facecolors='none', edgecolors='#d62728',
                linewidths=2, marker='o', label='Experimental', zorder=5)

ax2.set_xlabel('Liquid mole fraction, $x_{\\mathrm{IPA}}$ [-]', fontsize=13)
ax2.set_ylabel('Vapor mole fraction, $y_{\\mathrm{IPA}}$ [-]', fontsize=13)
ax2.set_aspect('equal')
ax2.set_xlim(-0.02, 1.02)
ax2.set_ylim(-0.02, 1.02)
ax2.tick_params(axis='both', which='major', labelsize=11, direction='in',
                length=6, width=1.5, top=True, right=True)
ax2.legend(loc='upper left', fontsize=10, frameon=True, fancybox=False,
           edgecolor='black', framealpha=1)
ax2.grid(True, linestyle='--', linewidth=0.5, alpha=0.3, color='gray')
ax2.text(0.98, 0.05, '(b)', transform=ax2.transAxes, fontsize=14,
         fontweight='bold', va='bottom', ha='right',
         bbox=dict(boxstyle='round,pad=0.3', facecolor='white', edgecolor='black', linewidth=1))

plt.tight_layout()
plt.savefig('isopropanol_water_vle_diagram.png', dpi=150, bbox_inches='tight',
            facecolor='white', edgecolor='none')
plt.savefig('isopropanol_water_vle_diagram.pdf', bbox_inches='tight',
            facecolor='white', edgecolor='none')
print('Saved figure as isopropanol_water_vle_diagram.png and .pdf')
plt.show()